[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S03/Reto_S03/laboratorio_automl/S3.5_lab_autogluon.ipynb)

# S3.5 · Laboratorio alternativo: AutoML sin nube de pago (AutoGluon)
**Sprint 3 · Reto "Primer modelo en producción" · IFCD107**

Equivale al laboratorio S3.4 (SageMaker), pero se ejecuta en Google Colab o en tu PC y **no tiene coste**.
Objetivo: predecir si una reserva de *TurisData Canarias* se cancelará.

Uso en Colab: ejecuta las celdas **de arriba abajo, de una en una** (Mayús+Intro). La primera instala AutoGluon (tarda 2–3 min).
Guía completa para trabajar en pareja: `sprints/S03/actividades/automl/guia_automl_en_parejas.md`.

## Instalación (solo Colab)
Si Colab os avisa de que hay que reiniciar el entorno, aceptad y seguid desde la celda **0. Configuración**.

In [ ]:
!pip install -q "autogluon.tabular[lightgbm,xgboost,catboost]" fastapi httpx

## 0. Configuración

In [ ]:
# Descarga de archivos del laboratorio (si no están ya en la carpeta de trabajo)
import os, urllib.request
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/sprints/S03/Reto_S03/laboratorio_automl/"
def archivo(nombre):
    """Devuelve el nombre del archivo; si no está en esta carpeta, lo descarga del repositorio del curso."""
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(BASE_URL + nombre, nombre)
    return nombre

import time

import numpy as np
import pandas as pd

RUTA_DATOS = archivo("reservas_turisdata.csv")
LIMITE_TIEMPO = 120   # segundos de entrenamiento (poned 15 para un ensayo rápido)
CARPETA_MODELO = "modelo_cancelaciones"
OBJETIVO = "cancelada"
SEMILLA = 42
np.random.seed(SEMILLA)

## 1. Cargar los datos

In [ ]:
# Si no tienes el CSV, genéralo con:  python generar_reservas_turisdata.py
datos = pd.read_csv(RUTA_DATOS, parse_dates=["fecha_reserva", "fecha_llegada"])
print(datos.shape)
print(datos.head())

## 2. Exploración y calidad del dato (equivale a "explorar el dataset" en SageMaker)

In [ ]:
print("\nTipos de dato:\n", datos.dtypes)
print("\nNulos por columna:\n", datos.isna().sum()[datos.isna().sum() > 0])
print("\nProporción de cancelaciones:", round(datos[OBJETIVO].mean(), 3))
print("\nResumen numérico:\n", datos.describe().T[["mean", "std", "min", "max"]])
# Preguntas para el informe:
#  - ¿Está desbalanceada la clase objetivo? ¿Qué métrica conviene?
#  - ¿Qué columnas podrían provocar fuga de datos (información posterior al desenlace)?

## 3. Partición TEMPORAL (evita fuga de datos: se entrena con el pasado y se prueba con el futuro)

In [ ]:
datos = datos.sort_values("fecha_reserva").reset_index(drop=True)
corte = int(len(datos) * 0.8)
entrenamiento = datos.iloc[:corte].drop(columns=["id_reserva"])
prueba = datos.iloc[corte:].drop(columns=["id_reserva"])
print(f"\nEntrenamiento: {len(entrenamiento)} filas ({entrenamiento.fecha_reserva.min().date()} a "
      f"{entrenamiento.fecha_reserva.max().date()})")
print(f"Prueba:        {len(prueba)} filas ({prueba.fecha_reserva.min().date()} a "
      f"{prueba.fecha_reserva.max().date()})")

## 4. Entrenar con AutoML (equivale al experimento de Autopilot/Canvas)

In [ ]:
from autogluon.tabular import TabularPredictor

inicio = time.time()
predictor = TabularPredictor(
    label=OBJETIVO,
    problem_type="binary",
    eval_metric="roc_auc",
    path=CARPETA_MODELO,
).fit(
    train_data=entrenamiento,
    time_limit=LIMITE_TIEMPO,
    presets="medium_quality",
)
tiempo_entrenamiento = time.time() - inicio
print(f"\nTiempo de entrenamiento: {tiempo_entrenamiento:.0f} s")

## 5. Revisar los modelos candidatos (leaderboard)

In [ ]:
clasificacion = predictor.leaderboard(prueba, silent=True)
print(clasificacion[["model", "score_test", "score_val", "fit_time"]].head(10))
# Preguntas: ¿el mejor modelo es un ensamblado? ¿Cuánto tarda cada uno? ¿Merece la pena el coste de cómputo?

## 6. Evaluar en el periodo de prueba

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

y_real = prueba[OBJETIVO]
prob = predictor.predict_proba(prueba.drop(columns=[OBJETIVO]))[1]  # probabilidad de cancelar
for umbral in (0.5, 0.3):
    pred = (prob >= umbral).astype(int)
    print(f"\nUmbral {umbral}: precisión={precision_score(y_real, pred):.3f} "
          f"recall={recall_score(y_real, pred):.3f} F1={f1_score(y_real, pred):.3f}")
print("\nROC-AUC:", round(roc_auc_score(y_real, prob), 3))
print("\nMatriz de confusión (umbral 0.5):\n", confusion_matrix(y_real, (prob >= 0.5).astype(int)))
print("\n", classification_report(y_real, (prob >= 0.5).astype(int), digits=3))
# Pregunta de negocio: ¿qué cuesta más, una cancelación no prevista o una alerta falsa? ¿Qué umbral elegirías?

## 6b. Elegir el umbral con costes (EJEMPLO: cambia los dos importes por los vuestros)

In [ ]:
COSTE_NO_PREVISTA = 100    # euros que cuesta una cancelación que no avisamos (noche vacía)
COSTE_ALERTA_FALSA = 40    # euros que cuesta una alerta que no hacía falta (p. ej. un descuento regalado)
y_array = y_real.to_numpy()
filas = []
for u in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    pred = (prob >= u).astype(int)
    no_previstas = int(((pred == 0) & (y_array == 1)).sum())
    falsas = int(((pred == 1) & (y_array == 0)).sum())
    filas.append({"umbral": u, "alertas": int(pred.sum()), "no_previstas": no_previstas,
                  "alertas_falsas": falsas, "coste_euros": COSTE_NO_PREVISTA * no_previstas + COSTE_ALERTA_FALSA * falsas})
tabla_costes = pd.DataFrame(filas)
print("\n", tabla_costes.to_string(index=False))
print("\nUmbral con menos coste (de los probados):", tabla_costes.loc[tabla_costes["coste_euros"].idxmin(), "umbral"])
print("Umbral teórico = alerta falsa / (alerta falsa + no prevista) =",
      round(COSTE_ALERTA_FALSA / (COSTE_ALERTA_FALSA + COSTE_NO_PREVISTA), 2))
print("Comparación: no avisar nunca =", COSTE_NO_PREVISTA * int(y_array.sum()), "euros · avisar a todos =",
      COSTE_ALERTA_FALSA * int((y_array == 0).sum()), "euros")
# Pregunta: ¿gana el modelo a «avisar a todos»? ¿Y si la alerta falsa costara 5 euros en vez de 40?

## 7. Explicabilidad: importancia de variables

In [ ]:
importancia = predictor.feature_importance(prueba, num_shuffle_sets=3, subsample_size=1000, silent=True)
print(importancia[["importance", "stddev"]].head(10))
# Pregunta: ¿tienen sentido las variables más importantes para el negocio? ¿Alguna sospechosa de fuga?

## 8. Guardar y recargar el modelo (equivale a "desplegar"; el modelo queda en una carpeta)

In [ ]:
predictor.save()
predictor_recargado = TabularPredictor.load(CARPETA_MODELO)
comprobacion = predictor_recargado.predict_proba(prueba.drop(columns=[OBJETIVO]).head(5))[1]
print("\nModelo recargado. Probabilidades de ejemplo:", comprobacion.round(3).tolist())

## 9. Función de predicción (la usará la API del paso 10)

In [ ]:
def predecir_cancelacion(reserva: dict) -> dict:
    """Recibe una reserva (dict con las columnas de entrada) y devuelve la probabilidad de cancelación."""
    fila = pd.DataFrame([reserva])
    fila["fecha_reserva"] = pd.to_datetime(fila["fecha_reserva"])
    fila["fecha_llegada"] = pd.to_datetime(fila["fecha_llegada"])
    p = float(predictor_recargado.predict_proba(fila)[1].iloc[0])
    return {"probabilidad_cancelacion": round(p, 4), "alerta": p >= 0.5}


ejemplo = prueba.drop(columns=[OBJETIVO]).iloc[0].to_dict()
print("\nEjemplo de predicción:", predecir_cancelacion(ejemplo))

## 10. Servir el modelo por API (equivale al endpoint)

In [ ]:
# En Colab no se puede abrir http://127.0.0.1:8000/docs desde tu navegador: usamos TestClient,
# que llama a la API "por dentro", igual que lo haría un cliente. En local: uvicorn api_cancelaciones:app --reload
import importlib, sys
archivo("api_cancelaciones.py")
sys.path.insert(0, os.getcwd())
api = importlib.import_module("api_cancelaciones")
from fastapi.testclient import TestClient
cliente = TestClient(api.app)
print(cliente.get("/salud").json())
reserva_ejemplo = {"fecha_reserva": "2025-11-03", "fecha_llegada": "2026-01-15", "antelacion_dias": 73, "noches": 5,
                   "adultos": 2, "ninos": 0, "tipo_habitacion": "estandar", "regimen": "desayuno",
                   "canal": "web_intermediaria", "pais_origen": "UK", "tarifa": "flexible",
                   "deposito": "sin_deposito", "precio_noche": 85.0, "peticiones_especiales": 0,
                   "cliente_repetidor": 0, "cancelaciones_previas": 0}
respuesta = cliente.post("/predecir", json=reserva_ejemplo)
print(respuesta.status_code, respuesta.json())
# Prueba también una reserva con datos imposibles (por ejemplo "noches": 0) y mira el error 422.

## 11. Informe de recursos (equivale al "informe de costes"; aquí el coste económico es 0 EUR)

In [ ]:
tamano_mb = sum(
    os.path.getsize(os.path.join(raiz, f))
    for raiz, _, ficheros in os.walk(CARPETA_MODELO) for f in ficheros
) / 1e6
print("\n--- Informe de recursos ---")
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.0f} s (límite fijado: {LIMITE_TIEMPO} s)")
print(f"Tamaño del modelo en disco: {tamano_mb:.1f} MB")
print("Coste económico: 0 EUR (Colab gratuito o equipo local)")
print("Recursos que hay que cerrar: la sesión de Colab (Entorno de ejecución > Desconectar).")